# Setup Landing Data: Open-Meteo Air Quality Telemetry & EPA Reference Standards

Prepares the landing storage:
1. Fetches real-time hourly air quality telemetry across global reference cities (New York, London, Tokyo, Berlin, Paris) via Open-Meteo Air Quality API.
2. Writes raw observations into landing JSON files split across batches.
3. Generates EPA Air Quality Index (AQI) classification standards as a reference CSV file.

In [0]:
import os
import json
import shutil
import urllib.request
import pandas as pd
from pyspark.sql import functions as F

In [0]:
# Configurable Widgets for Target Storage, Catalog Routing & Optional Custom Landing Path
dbutils.widgets.text("catalog", "dbr_dev", "Catalog")
dbutils.widgets.text("schema", "valeriimatviiv_bronze", "Schema")
dbutils.widgets.text("volume", "air_quality_landing", "Volume / Storage Name")
dbutils.widgets.text("custom_landing_path", "", "Custom Landing Path (Optional Override)")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
volume = dbutils.widgets.get("volume").strip()
custom_landing_path = dbutils.widgets.get("custom_landing_path").strip()

def is_catalog_available(cat_name):
    if not cat_name:
        return False
    try:
        available_catalogs = [row[0] for row in spark.sql("SHOW CATALOGS").collect()]
        return cat_name in available_catalogs
    except Exception:
        return False

if custom_landing_path:
    base_path = custom_landing_path.rstrip("/")
elif is_catalog_available(catalog):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")
    base_path = f"/Volumes/{catalog}/{schema}/{volume}"
else:
    try:
        current_user = spark.sql("SELECT current_user()").collect()[0][0]
        base_path = f"/Workspace/Users/{current_user}/{volume}"
    except Exception:
        base_path = f"/tmp/{volume}"
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")

landing_stream_path = f"{base_path}/landing/telemetry_stream"
landing_reference_path = f"{base_path}/landing/reference"

os.makedirs(f"{landing_stream_path}/batch_1", exist_ok=True)
os.makedirs(f"{base_path}/landing/staging/batch_2", exist_ok=True)
os.makedirs(landing_reference_path, exist_ok=True)
# Clean up any legacy staging inside stream directory to prevent duplicate file ingestion
shutil.rmtree(f"{landing_stream_path}/batch_2_staging", ignore_errors=True)


In [0]:
cities = [
    {"city": "New York", "country": "USA", "lat": 40.7128, "lon": -74.0060, "station_id": "US-NYC-001"},
    {"city": "London", "country": "UK", "lat": 51.5074, "lon": -0.1278, "station_id": "GB-LON-001"},
    {"city": "Tokyo", "country": "Japan", "lat": 35.6762, "lon": 139.6503, "station_id": "JP-TYO-001"},
    {"city": "Berlin", "country": "Germany", "lat": 52.5200, "lon": 13.4050, "station_id": "DE-BER-001"},
    {"city": "Paris", "country": "France", "lat": 48.8566, "lon": 2.3522, "station_id": "FR-PAR-001"}
]

records_batch_1 = []
records_batch_2 = []

for city_meta in cities:
    try:
        url = f"https://air-quality-api.open-meteo.com/v1/air-quality?latitude={city_meta['lat']}&longitude={city_meta['lon']}&hourly=pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,us_aqi&past_days=2"
        req = urllib.request.Request(url, headers={"User-Agent": "Databricks-Lakeflow/1.0"})
        with urllib.request.urlopen(req, timeout=10) as response:
            data = json.loads(response.read().decode())
            
        hourly = data.get("hourly", {})
        timestamps = hourly.get("time", [])
        pm10 = hourly.get("pm10", [])
        pm2_5 = hourly.get("pm2_5", [])
        co = hourly.get("carbon_monoxide", [])
        no2 = hourly.get("nitrogen_dioxide", [])
        so2 = hourly.get("sulphur_dioxide", [])
        o3 = hourly.get("ozone", [])
        us_aqi = hourly.get("us_aqi", [])
        
        total_points = len(timestamps)
        split_idx = total_points // 2
        
        for i in range(total_points):
            rec = {
                "event_id": f"{city_meta['station_id']}_{timestamps[i]}",
                "station_id": city_meta["station_id"],
                "city": city_meta["city"],
                "country": city_meta["country"],
                "latitude": float(city_meta["lat"]),
                "longitude": float(city_meta["lon"]),
                "timestamp": timestamps[i],
                "pm10": float(pm10[i]) if pm10[i] is not None else None,
                "pm2_5": float(pm2_5[i]) if pm2_5[i] is not None else None,
                "carbon_monoxide": float(co[i]) if co[i] is not None else None,
                "nitrogen_dioxide": float(no2[i]) if no2[i] is not None else None,
                "sulphur_dioxide": float(so2[i]) if so2[i] is not None else None,
                "ozone": float(o3[i]) if o3[i] is not None else None,
                "us_aqi": int(us_aqi[i]) if us_aqi[i] is not None else None
            }
            if i < split_idx:
                records_batch_1.append(rec)
            else:
                records_batch_2.append(rec)
    except Exception:
        pass

In [0]:
batch_1_file = f"{landing_stream_path}/batch_1/events.json"
with open(batch_1_file, "w") as f:
    for r in records_batch_1:
        f.write(json.dumps(r) + "\n")

batch_2_file = f"{base_path}/landing/staging/batch_2/events.json"
with open(batch_2_file, "w") as f:
    for r in records_batch_2:
        f.write(json.dumps(r) + "\n")

In [0]:
aqi_reference_data = [
    {"aqi_min": 0, "aqi_max": 50, "category": "Good", "color_code": "Green", "health_implication": "Air quality is satisfactory, poses little or no risk.", "cautionary_statement": "None."},
    {"aqi_min": 51, "aqi_max": 100, "category": "Moderate", "color_code": "Yellow", "health_implication": "Air quality is acceptable; some pollutants may cause moderate concern for sensitive individuals.", "cautionary_statement": "Unusually sensitive people should consider reducing prolonged outdoor exertion."},
    {"aqi_min": 101, "aqi_max": 150, "category": "Unhealthy for Sensitive Groups", "color_code": "Orange", "health_implication": "Members of sensitive groups may experience health effects.", "cautionary_statement": "People with respiratory or heart disease should limit prolonged outdoor exertion."},
    {"aqi_min": 151, "aqi_max": 200, "category": "Unhealthy", "color_code": "Red", "health_implication": "Everyone may begin to experience health effects; sensitive groups more serious effects.", "cautionary_statement": "Everyone should avoid prolonged outdoor exertion."},
    {"aqi_min": 201, "aqi_max": 300, "category": "Very Unhealthy", "color_code": "Purple", "health_implication": "Health alert: risk of health effects is increased for everyone.", "cautionary_statement": "Active children and adults should avoid all outdoor exertion."},
    {"aqi_min": 301, "aqi_max": 500, "category": "Hazardous", "color_code": "Maroon", "health_implication": "Health warning of emergency conditions: everyone is more likely to be affected.", "cautionary_statement": "Everyone should avoid all outdoor physical activity."}
]

csv_file_path = f"{landing_reference_path}/aqi_reference.csv"
df_pandas_ref = pd.DataFrame(aqi_reference_data)
df_pandas_ref.to_csv(csv_file_path, index=False)

df_reference = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(landing_reference_path)
# display(df_reference)

### Step 4 (Streaming Simulation): Release Batch 2 to Test Incremental Ingestion

In [0]:
src_batch_2 = f"{base_path}/landing/staging/batch_2/events.json"
dst_batch_2 = f"{landing_stream_path}/batch_2/events.json"

os.makedirs(f"{landing_stream_path}/batch_2", exist_ok=True)
shutil.copyfile(src_batch_2, dst_batch_2)